In [0]:
# =============================================================
# BASELINE CREDIT-RISK MODEL
#
# Model:
#   Median imputation
#   Robust scaling
#   Logistic regression  (class_weight="balanced" — required
#   because the bankrupt class is ~7% of training rows)
#
# Data:
#   Train      -> fit model and preprocessing
#   Validation -> initial evaluation
#   Test       -> intentionally untouched
# =============================================================

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
SPLIT_DIR = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/splits"
)

TRAIN_FILE      = f"{SPLIT_DIR}/train.parquet"
VALIDATION_FILE = f"{SPLIT_DIR}/validation.parquet"

TARGET_COLUMN = "bankrupt_within_1_year"

EXCLUDED_COLUMNS = {
    "borrower_id",
    "dataset_split",
    TARGET_COLUMN,
}

# ---------------------------------------------------------
# 2. Load only train and validation
# ---------------------------------------------------------
train_df      = pd.read_parquet(TRAIN_FILE)
validation_df = pd.read_parquet(VALIDATION_FILE)

# Safety: train and validation borrowers must not overlap
assert set(train_df["borrower_id"]).isdisjoint(
    set(validation_df["borrower_id"])
)

feature_columns = [
    col for col in train_df.columns
    if col not in EXCLUDED_COLUMNS
]

assert len(feature_columns) == 64, (
    f"Expected 64 model features, found {len(feature_columns)}"
)

X_train      = train_df[feature_columns]
y_train      = train_df[TARGET_COLUMN].astype(int)

X_validation = validation_df[feature_columns]
y_validation = validation_df[TARGET_COLUMN].astype(int)

print(f"Train rows       : {len(X_train):,}")
print(f"Validation rows  : {len(X_validation):,}")
print(f"Features         : {len(feature_columns)}")
print(f"Train event rate : {y_train.mean():.2%}")
print(f"Valid event rate : {y_validation.mean():.2%}")

# ---------------------------------------------------------
# 3. Define leakage-safe pipeline
#
# class_weight='balanced' adjusts for the ~7% minority class.
# All preprocessing is fitted only on X_train.
# ---------------------------------------------------------
baseline_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            RobustScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=5000,
                random_state=42,
                class_weight="balanced",  # required: bankrupt class ~7%
            ),
        ),
    ]
)

# ---------------------------------------------------------
# 4. Configure MLflow
# ---------------------------------------------------------
MLFLOW_EXPERIMENT = (
    "/Shared/credit-risk-copilot-model-development"
)

mlflow.set_experiment(MLFLOW_EXPERIMENT)

mlflow.sklearn.autolog(
    log_models=True,
    log_input_examples=True,
    log_model_signatures=True,
    log_datasets=True,
    silent=True,
)

# ---------------------------------------------------------
# 5. Train and evaluate
# ---------------------------------------------------------
with mlflow.start_run(
    run_name="logistic-regression-baseline"
) as run:

    baseline_pipeline.fit(X_train, y_train)

    validation_probability = (
        baseline_pipeline.predict_proba(X_validation)[:, 1]
    )

    # Initial threshold only.
    # Threshold optimisation is a later step.
    threshold = 0.50

    validation_prediction = (
        validation_probability >= threshold
    ).astype(int)

    metrics = {
        "validation_roc_auc": roc_auc_score(
            y_validation, validation_probability
        ),
        "validation_pr_auc": average_precision_score(
            y_validation, validation_probability
        ),
        "validation_brier_score": brier_score_loss(
            y_validation, validation_probability
        ),
        "validation_log_loss": log_loss(
            y_validation, validation_probability
        ),
        "validation_precision_at_0_5": precision_score(
            y_validation, validation_prediction, zero_division=0
        ),
        "validation_recall_at_0_5": recall_score(
            y_validation, validation_prediction, zero_division=0
        ),
        "validation_f1_at_0_5": f1_score(
            y_validation, validation_prediction, zero_division=0
        ),
    }

    mlflow.log_metrics(metrics)

    mlflow.log_params({
        "target_column":        TARGET_COLUMN,
        "feature_count":        len(feature_columns),
        "validation_threshold": threshold,
        "test_set_used":        False,
        "class_weight":         "balanced",
        "model_purpose": "Initial explainable bankruptcy-risk baseline",
    })

    mlflow.set_tags({
        "project":               "explainable-credit-risk-copilot",
        "model_stage":           "baseline",
        "data_source":           "UCI Polish Companies Bankruptcy",
        "human_review_required": "true",
    })

    tn, fp, fn, tp = confusion_matrix(
        y_validation, validation_prediction
    ).ravel()

    mlflow.log_metrics({
        "validation_true_negatives":  int(tn),
        "validation_false_positives": int(fp),
        "validation_false_negatives": int(fn),
        "validation_true_positives":  int(tp),
    })

    run_id = run.info.run_id

# ---------------------------------------------------------
# 6. Display results
# ---------------------------------------------------------
print(f"\nMLflow experiment: {MLFLOW_EXPERIMENT}")
print(f"MLflow run ID    : {run_id}")

print("\nValidation metrics:")
for name, value in metrics.items():
    print(f"  {name:36s}: {value:.4f}")

print("\nValidation confusion matrix at threshold 0.50:")
print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")

print(
    "\nTest set remains untouched: "
    "/splits/test.parquet"
)

In [0]:
# =============================================================
# CALIBRATED LOGISTIC-REGRESSION BASELINE
#
# Calibration:
#   - Sigmoid / Platt scaling
#   - Five-fold stratified CV using training data only
#
# Validation:
#   - Compare calibrated and uncalibrated probabilities
#
# Test:
#   - Remains completely untouched
# =============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from mlflow.models import infer_signature

# ---------------------------------------------------------
# 1. Confirm required objects exist
# ---------------------------------------------------------
required_objects = [
    "baseline_pipeline",
    "X_train", "y_train",
    "X_validation", "y_validation",
]

for obj_name in required_objects:
    assert obj_name in globals(), (
        f"{obj_name} is missing. "
        "Run the baseline-model cell first."
    )

# Disable autolog before any .fit() calls so that the fold-level
# and clone fits do not create spurious MLflow auto-runs from the
# autolog session opened in cell 1.
mlflow.sklearn.autolog(disable=True)

# ---------------------------------------------------------
# 2. Uncalibrated validation probabilities
# ---------------------------------------------------------
uncalibrated_model = clone(baseline_pipeline)
uncalibrated_model.fit(X_train, y_train)

uncalibrated_probability = (
    uncalibrated_model.predict_proba(X_validation)[:, 1]
)

# ---------------------------------------------------------
# 3. Create calibrated model
#
# Each training fold fits preprocessing and LR together,
# then produces unbiased probabilities for the calibrator.
# Validation data is never used here.
# ---------------------------------------------------------
calibration_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

calibrated_model = CalibratedClassifierCV(
    estimator=clone(baseline_pipeline),
    method="sigmoid",
    cv=calibration_cv,
    ensemble=True,
    n_jobs=-1,
)

calibrated_model.fit(X_train, y_train)

calibrated_probability = (
    calibrated_model.predict_proba(X_validation)[:, 1]
)

# ---------------------------------------------------------
# 4. Evaluation helper
# ---------------------------------------------------------
def calculate_probability_metrics(y_true, probability, threshold=0.50):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction).ravel()
    return {
        "roc_auc":          roc_auc_score(y_true, probability),
        "pr_auc":           average_precision_score(y_true, probability),
        "brier_score":      brier_score_loss(y_true, probability),
        "log_loss":         log_loss(y_true, probability),
        "precision_at_0_5": precision_score(y_true, prediction, zero_division=0),
        "recall_at_0_5":    recall_score(y_true, prediction, zero_division=0),
        "f1_at_0_5":        f1_score(y_true, prediction, zero_division=0),
        "true_negatives":   int(tn),
        "false_positives":  int(fp),
        "false_negatives":  int(fn),
        "true_positives":   int(tp),
    }


uncalibrated_metrics = calculate_probability_metrics(
    y_validation, uncalibrated_probability
)
calibrated_metrics = calculate_probability_metrics(
    y_validation, calibrated_probability
)

# ---------------------------------------------------------
# 5. Base-rate reference model
# ---------------------------------------------------------
validation_event_rate = float(y_validation.mean())

base_rate_probability = np.full(
    shape=len(y_validation),
    fill_value=validation_event_rate,
)

base_rate_brier    = brier_score_loss(y_validation, base_rate_probability)
base_rate_log_loss = log_loss(y_validation, base_rate_probability)

# ---------------------------------------------------------
# 6. Calibration plot
# ---------------------------------------------------------
CALIBRATION_PLOT = "/tmp/calibration_comparison.png"

fig, ax = plt.subplots(figsize=(7, 6))

CalibrationDisplay.from_predictions(
    y_validation, uncalibrated_probability,
    n_bins=10, strategy="quantile",
    name="Uncalibrated logistic regression",
    ax=ax,
)
CalibrationDisplay.from_predictions(
    y_validation, calibrated_probability,
    n_bins=10, strategy="quantile",
    name="Sigmoid-calibrated logistic regression",
    ax=ax,
)
ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
ax.set_title("Validation probability calibration")
ax.legend()
fig.tight_layout()
fig.savefig(CALIBRATION_PLOT, dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------
# 7. Log calibrated run to MLflow
# ---------------------------------------------------------
MLFLOW_EXPERIMENT = "/Shared/credit-risk-copilot-model-development"
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(
    run_name="logistic-regression-sigmoid-calibrated"
) as run:

    mlflow.log_params({
        "base_model":                     "logistic_regression",
        "calibration_method":             "sigmoid",
        "calibration_cv_folds":           5,
        "calibration_cv_stratified":      True,
        "calibration_uses_training_only": True,
        "test_set_used":                  False,
        "feature_count":                  X_train.shape[1],
    })

    mlflow.log_metrics({
        **{f"validation_calibrated_{k}": v   for k, v in calibrated_metrics.items()},
        **{f"validation_uncalibrated_{k}": v for k, v in uncalibrated_metrics.items()},
        "validation_base_rate":           validation_event_rate,
        "validation_base_rate_brier":     base_rate_brier,
        "validation_base_rate_log_loss":  base_rate_log_loss,
    })

    mlflow.set_tags({
        "project":               "explainable-credit-risk-copilot",
        "model_stage":           "probability-calibration",
        "model_type":            "calibrated-logistic-regression",
        "human_review_required": "true",
    })

    input_example         = X_train.head(5)
    model_output_example  = calibrated_model.predict_proba(input_example)[:, 1]
    signature             = infer_signature(input_example, model_output_example)

    mlflow.sklearn.log_model(
        sk_model=calibrated_model,
        artifact_path="model",
        input_example=input_example,
        signature=signature,
    )

    mlflow.log_artifact(CALIBRATION_PLOT, artifact_path="evaluation")

    calibrated_run_id = run.info.run_id

# Re-enable autolog for later cells
mlflow.sklearn.autolog(
    log_models=True,
    log_input_examples=True,
    log_model_signatures=True,
    silent=True,
)

# ---------------------------------------------------------
# 8. Compare results
# ---------------------------------------------------------
comparison_df = pd.DataFrame([
    {
        "model":       "Base-rate reference",
        "roc_auc":     0.5000,
        "pr_auc":      validation_event_rate,
        "brier_score": base_rate_brier,
        "log_loss":    base_rate_log_loss,
    },
    {
        "model":       "Uncalibrated logistic regression",
        "roc_auc":     uncalibrated_metrics["roc_auc"],
        "pr_auc":      uncalibrated_metrics["pr_auc"],
        "brier_score": uncalibrated_metrics["brier_score"],
        "log_loss":    uncalibrated_metrics["log_loss"],
    },
    {
        "model":       "Sigmoid-calibrated logistic regression",
        "roc_auc":     calibrated_metrics["roc_auc"],
        "pr_auc":      calibrated_metrics["pr_auc"],
        "brier_score": calibrated_metrics["brier_score"],
        "log_loss":    calibrated_metrics["log_loss"],
    },
])

print(f"MLflow run ID: {calibrated_run_id}")

print("\nProbability-quality comparison:")
display(comparison_df.round(4))

print("\nCalibrated threshold metrics at 0.50:")
for key in [
    "precision_at_0_5", "recall_at_0_5", "f1_at_0_5",
    "false_positives", "false_negatives", "true_positives",
]:
    print(f"  {key:24s}: {calibrated_metrics[key]}")

print(
    "\nTest set remains untouched: "
    "/splits/test.parquet"
)

In [0]:
# =============================================================
# VALIDATION-ONLY DECISION THRESHOLD SELECTION
#
# Objective:
#   Recall at least 80% of bankruptcy cases.
#   Among eligible thresholds, choose the highest precision.
#
# Test set remains untouched.
# =============================================================

import numpy as np
import pandas as pd
import mlflow

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)

TARGET_RECALL = 0.80

# calibrated_probability and y_validation come from
# the previous calibration cell.
assert "calibrated_probability" in globals()
assert "y_validation" in globals()

# ---------------------------------------------------------
# 1. Calculate precision and recall at all thresholds
#
# precision_recall_curve returns arrays of length n+1;
# thresholds has length n, so use [:-1] to align.
# ---------------------------------------------------------
precision_values, recall_values, thresholds = precision_recall_curve(
    y_validation,
    calibrated_probability,
)

threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision_values[:-1],
    "recall":    recall_values[:-1],
})

threshold_results["f1"] = (
    2
    * threshold_results["precision"]
    * threshold_results["recall"]
    / (
        threshold_results["precision"]
        + threshold_results["recall"]
    ).replace(0, np.nan)
)

# ---------------------------------------------------------
# 2. Keep thresholds satisfying the recall requirement
# ---------------------------------------------------------
eligible_thresholds = threshold_results[
    threshold_results["recall"] >= TARGET_RECALL
].copy()

assert not eligible_thresholds.empty, (
    f"No threshold achieves recall >= {TARGET_RECALL:.0%}"
)

# Highest precision among thresholds meeting recall target.
# Prefer highest threshold as tiebreaker to minimise review load.
selected_row = (
    eligible_thresholds
    .sort_values(by=["precision", "threshold"], ascending=[False, False])
    .iloc[0]
)

selected_threshold = float(selected_row["threshold"])

# ---------------------------------------------------------
# 3. Evaluate the selected operating point
# ---------------------------------------------------------
selected_predictions = (
    calibrated_probability >= selected_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_validation, selected_predictions
).ravel()

selected_metrics = {
    "selected_threshold":          selected_threshold,
    "validation_precision":        precision_score(y_validation, selected_predictions, zero_division=0),
    "validation_recall":           recall_score(y_validation, selected_predictions, zero_division=0),
    "validation_f1":               f1_score(y_validation, selected_predictions, zero_division=0),
    "validation_balanced_accuracy": balanced_accuracy_score(y_validation, selected_predictions),
    "validation_true_negatives":   int(tn),
    "validation_false_positives":  int(fp),
    "validation_false_negatives":  int(fn),
    "validation_true_positives":   int(tp),
    "validation_review_rate":      float(selected_predictions.mean()),
}

# ---------------------------------------------------------
# 4. Log threshold to the calibrated MLflow run
# ---------------------------------------------------------
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(
    run_id=calibrated_run_id,  # append to the calibrated run
):
    mlflow.log_metrics({
        f"threshold_{k}": v
        for k, v in selected_metrics.items()
        if isinstance(v, (int, float))
    })
    mlflow.log_param("recall_policy", f"recall>={TARGET_RECALL:.0%}_max_precision")

# ---------------------------------------------------------
# 5. Display nearby operating points
# ---------------------------------------------------------
nearby_thresholds = (
    threshold_results
    .assign(distance=lambda df: abs(df["threshold"] - selected_threshold))
    .sort_values("distance")
    .head(10)
    .drop(columns="distance")
    .sort_values("threshold")
)

print(f"Target recall      : {TARGET_RECALL:.0%}")
print(f"Selected threshold : {selected_threshold:.4f}")

print("\nSelected validation operating point:")
for name, value in selected_metrics.items():
    if isinstance(value, float):
        print(f"  {name:32s}: {value:.4f}")
    else:
        print(f"  {name:32s}: {value}")

print("\nNearby thresholds:")
display(nearby_thresholds.round(4))

print(
    "\nTest set remains untouched: "
    "/splits/test.parquet"
)

In [0]:
# =============================================================
# V2 LOCKED MODEL CANDIDATE
#
# 1. Train logistic-regression pipeline on V2 train
# 2. Calibrate probabilities using training-only 5-fold CV
# 3. Select threshold using V2 validation only
# 4. Log model, metrics and threshold to MLflow
#
# V2 test remains unopened.
# =============================================================

import json
import os
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)
from mlflow.models import infer_signature

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
CLEAN_SPLIT_DIR = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/splits_v2"
)

TRAIN_FILE      = f"{CLEAN_SPLIT_DIR}/train.parquet"
VALIDATION_FILE = f"{CLEAN_SPLIT_DIR}/validation.parquet"
TEST_FILE       = f"{CLEAN_SPLIT_DIR}/test.parquet"  # do not read

TARGET_COLUMN   = "bankrupt_within_1_year"
TARGET_RECALL   = 0.80

EXCLUDED_COLUMNS = {
    "borrower_id",
    "dataset_split",
    TARGET_COLUMN,
}

MLFLOW_EXPERIMENT = "/Shared/credit-risk-copilot-model-development"

THRESHOLD_OUTPUT = f"{CLEAN_SPLIT_DIR}/locked_model_configuration_v2.json"

# ---------------------------------------------------------
# 2. Load only train and validation
# ---------------------------------------------------------
train_v2      = pd.read_parquet(TRAIN_FILE)
validation_v2 = pd.read_parquet(VALIDATION_FILE)

feature_columns = [
    col for col in train_v2.columns
    if col not in EXCLUDED_COLUMNS
]

assert len(feature_columns) == 64
assert set(train_v2["borrower_id"]).isdisjoint(set(validation_v2["borrower_id"]))

X_train_v2      = train_v2[feature_columns]
y_train_v2      = train_v2[TARGET_COLUMN].astype(int)

X_validation_v2 = validation_v2[feature_columns]
y_validation_v2 = validation_v2[TARGET_COLUMN].astype(int)

print(f"Train rows       : {len(X_train_v2):,}")
print(f"Validation rows  : {len(X_validation_v2):,}")
print(f"Features         : {len(feature_columns)}")
print(f"Train event rate : {y_train_v2.mean():.2%}")
print(f"Valid event rate : {y_validation_v2.mean():.2%}")

# ---------------------------------------------------------
# 3. Define base pipeline
#
# class_weight='balanced' required: positive-class rate is
# 6.80%, well below the 20% threshold for balanced training.
# Imputation and scaling are fitted inside each CV fold.
# ---------------------------------------------------------
base_pipeline_v2 = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True),
    ),
    (
        "scaler",
        RobustScaler(),
    ),
    (
        "classifier",
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        ),
    ),
])

# ---------------------------------------------------------
# 4. Fit sigmoid calibration using V2 train only
# ---------------------------------------------------------
# Disable autolog before any .fit() calls.
mlflow.sklearn.autolog(disable=True)

calibration_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

calibrated_model_v2 = CalibratedClassifierCV(
    estimator=base_pipeline_v2,
    method="sigmoid",
    cv=calibration_cv,
    ensemble=True,
    n_jobs=-1,
)

calibrated_model_v2.fit(X_train_v2, y_train_v2)

validation_probability_v2 = (
    calibrated_model_v2.predict_proba(X_validation_v2)[:, 1]
)

# ---------------------------------------------------------
# 5. Probability-quality metrics
# ---------------------------------------------------------
probability_metrics_v2 = {
    "validation_roc_auc":    roc_auc_score(y_validation_v2, validation_probability_v2),
    "validation_pr_auc":     average_precision_score(y_validation_v2, validation_probability_v2),
    "validation_brier_score": brier_score_loss(y_validation_v2, validation_probability_v2),
    "validation_log_loss":   log_loss(y_validation_v2, validation_probability_v2),
}

base_rate    = float(y_validation_v2.mean())
base_rate_brier    = brier_score_loss(y_validation_v2, np.full(len(y_validation_v2), base_rate))
base_rate_log_loss = log_loss(y_validation_v2, np.full(len(y_validation_v2), base_rate))

# ---------------------------------------------------------
# 6. Select validation threshold
#
# Rule: highest precision among thresholds achieving recall >= 80%.
# ---------------------------------------------------------
precision_values, recall_values, thresholds = precision_recall_curve(
    y_validation_v2, validation_probability_v2
)

threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision_values[:-1],
    "recall":    recall_values[:-1],
})

threshold_table["f1"] = (
    2
    * threshold_table["precision"]
    * threshold_table["recall"]
    / (threshold_table["precision"] + threshold_table["recall"]).replace(0, np.nan)
)

eligible_thresholds = threshold_table[
    threshold_table["recall"] >= TARGET_RECALL
].copy()

assert not eligible_thresholds.empty

selected_row = (
    eligible_thresholds
    .sort_values(["precision", "threshold"], ascending=[False, False])
    .iloc[0]
)

locked_threshold_v2 = float(selected_row["threshold"])

validation_prediction_v2 = (
    validation_probability_v2 >= locked_threshold_v2
).astype(int)

tn, fp, fn, tp = confusion_matrix(y_validation_v2, validation_prediction_v2).ravel()

operating_metrics_v2 = {
    "locked_threshold":            locked_threshold_v2,
    "validation_precision":        precision_score(y_validation_v2, validation_prediction_v2, zero_division=0),
    "validation_recall":           recall_score(y_validation_v2, validation_prediction_v2, zero_division=0),
    "validation_f1":               f1_score(y_validation_v2, validation_prediction_v2, zero_division=0),
    "validation_balanced_accuracy": balanced_accuracy_score(y_validation_v2, validation_prediction_v2),
    "validation_review_rate":      float(validation_prediction_v2.mean()),
    "validation_true_negatives":   int(tn),
    "validation_false_positives":  int(fp),
    "validation_false_negatives":  int(fn),
    "validation_true_positives":   int(tp),
}

# ---------------------------------------------------------
# 7. Save locked configuration as JSON
# ---------------------------------------------------------
locked_configuration = {
    "model_name":              "calibrated_logistic_regression_v2",
    "calibration_method":      "sigmoid",
    "calibration_folds":       5,
    "threshold_selection_rule": "maximize precision subject to recall >= 0.80",
    "target_recall":            TARGET_RECALL,
    "locked_threshold":         locked_threshold_v2,
    "feature_columns":          feature_columns,
    "train_file":               TRAIN_FILE,
    "validation_file":          VALIDATION_FILE,
    "test_file":                TEST_FILE,
    "test_set_used":            False,
}

with open(THRESHOLD_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(locked_configuration, f, indent=2)

# ---------------------------------------------------------
# 8. Log one complete V2 candidate run to MLflow
# ---------------------------------------------------------
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(
    run_name="v2-calibrated-logistic-locked-candidate"
) as run:

    mlflow.log_params({
        "model_type":                "logistic_regression",
        "calibration_method":        "sigmoid",
        "calibration_cv_folds":      5,
        "class_weight":              "balanced",
        "target_recall":             TARGET_RECALL,
        "threshold_selection_dataset": "validation_v2",
        "feature_count":             len(feature_columns),
        "demo_borrowers_excluded":   30,
        "test_set_used":             False,
    })

    mlflow.log_metrics({
        **probability_metrics_v2,
        **{k: v for k, v in operating_metrics_v2.items()
           if isinstance(v, (int, float))},
        "validation_base_rate":           base_rate,
        "validation_base_rate_brier":     base_rate_brier,
        "validation_base_rate_log_loss":  base_rate_log_loss,
    })

    mlflow.set_tags({
        "project":               "explainable-credit-risk-copilot",
        "model_stage":           "locked-candidate",
        "data_version":          "splits_v2",
        "threshold_is_bank_policy": "false",
        "human_review_required": "true",
    })

    input_example        = X_train_v2.head(5)
    model_output_example = calibrated_model_v2.predict_proba(input_example)[:, 1]
    signature            = infer_signature(input_example, model_output_example)

    mlflow.sklearn.log_model(
        sk_model=calibrated_model_v2,
        artifact_path="model",
        input_example=input_example,
        signature=signature,
    )

    mlflow.log_artifact(THRESHOLD_OUTPUT, artifact_path="configuration")

    locked_candidate_run_id = run.info.run_id

# Re-enable autolog for any subsequent cells
mlflow.sklearn.autolog(
    log_models=True,
    log_input_examples=True,
    log_model_signatures=True,
    silent=True,
)

# ---------------------------------------------------------
# 9. Results
# ---------------------------------------------------------
print(f"\nMLflow run ID      : {locked_candidate_run_id}")
print(f"Configuration file : {THRESHOLD_OUTPUT}")

print("\nProbability metrics:")
for name, value in probability_metrics_v2.items():
    print(f"  {name:32s}: {value:.4f}")
print(f"  {'base_rate_brier':32s}: {base_rate_brier:.4f}")
print(f"  {'base_rate_log_loss':32s}: {base_rate_log_loss:.4f}")

print("\nLocked validation operating point:")
for name, value in operating_metrics_v2.items():
    if isinstance(value, float):
        print(f"  {name:32s}: {value:.4f}")
    else:
        print(f"  {name:32s}: {value}")

print("\nV2 test remains unopened:", TEST_FILE)

In [0]:
# =============================================================
# SERVING-READY CREDIT-RISK MODEL
#
# Wraps the calibrated sklearn model and returns:
#   - calibrated probability
#   - locked review threshold
#   - deterministic human-review flag
#
# V2 test set remains unopened.
# =============================================================

import os
import numpy as np
import pandas as pd
import sklearn
import mlflow
import mlflow.pyfunc

from mlflow.models import infer_signature

# ---------------------------------------------------------
# 1. Confirm required V2 objects exist
# ---------------------------------------------------------
required_objects = [
    "calibrated_model_v2",
    "locked_threshold_v2",
    "feature_columns",
    "X_train_v2",
]

for object_name in required_objects:
    assert object_name in globals(), (
        f"{object_name} is missing. "
        "Run the V2 locked-candidate training cell first."
    )

SOURCE_CANDIDATE_RUN_ID = "e5db2fb9d9ac45c09143b47bc3755d21"
MLFLOW_EXPERIMENT = "/Shared/credit-risk-copilot-model-development"

# ---------------------------------------------------------
# 2. Define serving wrapper
# ---------------------------------------------------------
class CreditRiskServingModel(mlflow.pyfunc.PythonModel):

    def __init__(
        self,
        calibrated_model,
        feature_columns,
        review_threshold,
    ):
        self.calibrated_model = calibrated_model
        self.feature_columns   = list(feature_columns)
        self.review_threshold  = float(review_threshold)

    def predict(self, context, model_input, params=None):
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)

        missing_columns = [
            col for col in self.feature_columns
            if col not in model_input.columns
        ]
        if missing_columns:
            raise ValueError(
                "Missing required model features: "
                + ", ".join(missing_columns)
            )

        features = model_input[self.feature_columns].copy()

        probability = (
            self.calibrated_model
            .predict_proba(features)[:, 1]
            .astype(float)
        )

        review_required = probability >= self.review_threshold

        return pd.DataFrame({
            "probability_of_bankruptcy": probability,
            "review_threshold": np.full(
                len(probability), self.review_threshold, dtype=float
            ),
            "review_required": review_required.astype(bool),
            "decision_support_status": np.where(
                review_required,
                "REFER_FOR_HUMAN_REVIEW",
                "BELOW_REVIEW_THRESHOLD",
            ),
        })


serving_wrapper = CreditRiskServingModel(
    calibrated_model=calibrated_model_v2,
    feature_columns=feature_columns,
    review_threshold=locked_threshold_v2,
)

# ---------------------------------------------------------
# 3. Input / output example for signature inference
#
# Fill NaN with column medians so the stored JSON example
# round-trips cleanly. The pipeline still accepts missing
# values through its internal median imputer at serve time.
# ---------------------------------------------------------
input_example    = X_train_v2.head(5).copy()
training_medians = X_train_v2.median(numeric_only=True)
input_example    = input_example.fillna(training_medians)

output_example = serving_wrapper.predict(
    context=None, model_input=input_example
)

signature = infer_signature(input_example, output_example)

print("Serving input shape :", input_example.shape)
print("Serving output:")
display(output_example)

# ---------------------------------------------------------
# 4. Log serving-ready run to MLflow
# ---------------------------------------------------------
mlflow.sklearn.autolog(disable=True)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(
    run_name="v2-credit-risk-serving-pyfunc"
) as serving_run:

    mlflow.log_params({
        "source_candidate_run_id": SOURCE_CANDIDATE_RUN_ID,
        "base_model":              "calibrated_logistic_regression",
        "calibration_method":      "sigmoid",
        "review_threshold":        locked_threshold_v2,
        "threshold_selection_rule": "maximize precision subject to recall >= 0.80",
        "feature_count":           len(feature_columns),
        "data_version":            "splits_v2",
        "test_set_used":           False,
    })

    mlflow.set_tags({
        "project":                    "explainable-credit-risk-copilot",
        "model_stage":               "serving-ready-candidate",
        "output_contains_probability": "true",
        "human_review_required":      "true",
        "automatic_lending_decision": "false",
    })

    INPUT_EXAMPLE_FILE = "/tmp/credit_risk_serving_input_example.parquet"
    input_example.to_parquet(INPUT_EXAMPLE_FILE, index=False)
    mlflow.log_artifact(INPUT_EXAMPLE_FILE, artifact_path="examples")

    if "THRESHOLD_OUTPUT" in globals() and os.path.exists(THRESHOLD_OUTPUT):
        mlflow.log_artifact(THRESHOLD_OUTPUT, artifact_path="configuration")

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=serving_wrapper,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"scikit-learn=={sklearn.__version__}",
            f"pandas=={pd.__version__}",
            f"numpy=={np.__version__}",
        ],
    )

    serving_run_id    = serving_run.info.run_id
    serving_model_uri = model_info.model_uri

# ---------------------------------------------------------
# 5. Load the logged artifact and validate locally
# ---------------------------------------------------------
loaded_serving_model = mlflow.pyfunc.load_model(serving_model_uri)
loaded_predictions   = loaded_serving_model.predict(input_example)

native_probabilities = (
    calibrated_model_v2.predict_proba(input_example)[:, 1]
)

expected_output_columns = [
    "probability_of_bankruptcy",
    "review_threshold",
    "review_required",
    "decision_support_status",
]

assert list(loaded_predictions.columns) == expected_output_columns

assert np.allclose(
    loaded_predictions["probability_of_bankruptcy"].to_numpy(),
    native_probabilities,
    rtol=1e-10,
    atol=1e-12,
)

assert loaded_predictions["probability_of_bankruptcy"].between(0.0, 1.0).all()

assert np.allclose(
    loaded_predictions["review_threshold"].to_numpy(),
    locked_threshold_v2,
)

expected_review_flag = (
    loaded_predictions["probability_of_bankruptcy"].to_numpy()
    >= locked_threshold_v2
)
assert np.array_equal(
    loaded_predictions["review_required"].to_numpy(),
    expected_review_flag,
)

# ---------------------------------------------------------
# 6. Results
# ---------------------------------------------------------
print(f"Serving run ID    : {serving_run_id}")
print(f"Serving model URI : {serving_model_uri}")
print(f"Locked threshold  : {locked_threshold_v2:.4f}")
print("Local load test   : passed")
print("Probability check : passed")
print("Threshold check   : passed")
print("Test set used     : False")
display(loaded_predictions)

In [0]:
# =============================================================
# FINAL UNTOUCHED V2 TEST EVALUATION
#
# Evaluates:
#   - the exact logged serving PyFunc model
#   - the locked probability calibration
#   - the locked human-review threshold
#
# Important:
#   This is the single final test evaluation.
#   Do not tune the model or threshold afterward.
# =============================================================

import os
import json
import numpy as np
import pandas as pd

import mlflow
import mlflow.pyfunc

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)

# ---------------------------------------------------------
# 1. Frozen model and dataset configuration
# ---------------------------------------------------------
SERVING_RUN_ID      = "61921922ed4d487bbc1dd39da6b227ae"
SERVING_MODEL_URI   = f"runs:/{SERVING_RUN_ID}/model"

BASE_PATH = "/Volumes/demodatabrciks123/creditrisk/raw_file/structured"

TEST_FILE                  = f"{BASE_PATH}/splits_v2/test.parquet"
LOCKED_CONFIGURATION_FILE  = f"{BASE_PATH}/splits_v2/locked_model_configuration_v2.json"
FINAL_EVALUATION_DIR       = f"{BASE_PATH}/final_evaluation_v2"
PREDICTIONS_FILE           = f"{FINAL_EVALUATION_DIR}/test_predictions.parquet"
METRICS_FILE               = f"{FINAL_EVALUATION_DIR}/test_metrics.json"

MLFLOW_EXPERIMENT = "/Shared/credit-risk-copilot-model-development"
TARGET_COLUMN     = "bankrupt_within_1_year"

os.makedirs(FINAL_EVALUATION_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load the locked configuration
# ---------------------------------------------------------
with open(LOCKED_CONFIGURATION_FILE, "r", encoding="utf-8") as f:
    locked_configuration = json.load(f)

feature_columns  = locked_configuration["feature_columns"]
locked_threshold = float(locked_configuration["locked_threshold"])

assert len(feature_columns) == 64
assert locked_configuration["test_set_used"] is False

print(f"Serving model URI : {SERVING_MODEL_URI}")
print(f"Locked threshold  : {locked_threshold:.6f}")
print(f"Feature count     : {len(feature_columns)}")

# ---------------------------------------------------------
# 3. Open the V2 test set
# ---------------------------------------------------------
test_v2  = pd.read_parquet(TEST_FILE)

assert len(test_v2) == 882
assert test_v2[TARGET_COLUMN].sum() == 60
assert test_v2["borrower_id"].is_unique

missing_features = [
    col for col in feature_columns
    if col not in test_v2.columns
]
assert not missing_features, f"Missing test features: {missing_features}"

X_test_v2 = test_v2[feature_columns].copy()
y_test_v2 = test_v2[TARGET_COLUMN].astype(int)

print(f"\nTest rows       : {len(X_test_v2):,}")
print(f"Test positives  : {y_test_v2.sum()}")
print(f"Test event rate : {y_test_v2.mean():.2%}")

# ---------------------------------------------------------
# 4. Load and score the exact serving model
# ---------------------------------------------------------
serving_model  = mlflow.pyfunc.load_model(SERVING_MODEL_URI)
serving_output = serving_model.predict(X_test_v2)

required_output_columns = {
    "probability_of_bankruptcy",
    "review_threshold",
    "review_required",
    "decision_support_status",
}
assert required_output_columns.issubset(serving_output.columns)

test_probability     = serving_output["probability_of_bankruptcy"].astype(float).to_numpy()
test_review_required = serving_output["review_required"].astype(bool).to_numpy()
returned_threshold   = serving_output["review_threshold"].astype(float).to_numpy()

# ---------------------------------------------------------
# 5. Serving-contract validations
# ---------------------------------------------------------
assert np.isfinite(test_probability).all()
assert ((test_probability >= 0.0) & (test_probability <= 1.0)).all()
assert np.allclose(returned_threshold, locked_threshold, rtol=0, atol=1e-12)
assert np.array_equal(test_review_required, test_probability >= locked_threshold)

test_probability_clipped = np.clip(test_probability, 1e-15, 1 - 1e-15)

# ---------------------------------------------------------
# 6. Calculate final probability metrics
# ---------------------------------------------------------
test_event_rate       = float(y_test_v2.mean())
base_rate_probability = np.full(len(y_test_v2), test_event_rate)

tn, fp, fn, tp = confusion_matrix(y_test_v2, test_review_required).ravel()

final_test_metrics = {
    "test_roc_auc":                    float(roc_auc_score(y_test_v2, test_probability)),
    "test_pr_auc":                     float(average_precision_score(y_test_v2, test_probability)),
    "test_brier_score":                float(brier_score_loss(y_test_v2, test_probability)),
    "test_log_loss":                   float(log_loss(y_test_v2, test_probability_clipped)),
    "test_precision":                  float(precision_score(y_test_v2, test_review_required, zero_division=0)),
    "test_recall":                     float(recall_score(y_test_v2, test_review_required, zero_division=0)),
    "test_f1":                         float(f1_score(y_test_v2, test_review_required, zero_division=0)),
    "test_balanced_accuracy":          float(balanced_accuracy_score(y_test_v2, test_review_required)),
    "test_review_rate":                float(test_review_required.mean()),
    "test_event_rate":                 test_event_rate,
    "test_mean_predicted_probability": float(test_probability.mean()),
    "test_calibration_gap":            float(test_probability.mean() - test_event_rate),
    "test_base_rate_brier":            float(brier_score_loss(y_test_v2, base_rate_probability)),
    "test_base_rate_log_loss":         float(log_loss(y_test_v2, base_rate_probability)),
    "test_true_negatives":             int(tn),
    "test_false_positives":            int(fp),
    "test_false_negatives":            int(fn),
    "test_true_positives":             int(tp),
    "locked_threshold":                locked_threshold,
}

# ---------------------------------------------------------
# 7. Save record-level test predictions and metrics
# ---------------------------------------------------------
test_predictions = pd.DataFrame({
    "borrower_id":                  test_v2["borrower_id"].values,
    "actual_bankrupt_within_1_year": y_test_v2.values,
    "probability_of_bankruptcy":    test_probability,
    "locked_review_threshold":       locked_threshold,
    "review_required":              test_review_required,
    "decision_support_status":      serving_output["decision_support_status"].values,
})

test_predictions.to_parquet(PREDICTIONS_FILE, index=False)

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(final_test_metrics, f, indent=2)

# ---------------------------------------------------------
# 8. Log final evaluation to MLflow
# ---------------------------------------------------------
mlflow.sklearn.autolog(disable=True)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(
    run_name="v2-final-untouched-test-evaluation"
) as final_test_run:

    mlflow.log_params({
        "serving_model_run_id":               SERVING_RUN_ID,
        "serving_model_uri":                  SERVING_MODEL_URI,
        "data_version":                       "splits_v2",
        "evaluation_dataset":                 "test_v2",
        "test_rows":                          len(test_v2),
        "test_positive_cases":                int(y_test_v2.sum()),
        "locked_threshold":                   locked_threshold,
        "model_changed_after_validation":     False,
        "threshold_changed_after_validation": False,
    })

    mlflow.log_metrics(final_test_metrics)

    mlflow.set_tags({
        "project":                   "explainable-credit-risk-copilot",
        "model_stage":               "final-test-evaluation",
        "evaluation_is_final":       "true",
        "human_review_required":     "true",
        "automatic_lending_decision": "false",
    })

    mlflow.log_artifact(PREDICTIONS_FILE, artifact_path="test_evaluation")
    mlflow.log_artifact(METRICS_FILE,     artifact_path="test_evaluation")

    final_test_run_id = final_test_run.info.run_id

# ---------------------------------------------------------
# 9. Present final results
# ---------------------------------------------------------
print(f"\nFinal test run ID : {final_test_run_id}")
print(f"Predictions file  : {PREDICTIONS_FILE}")
print(f"Metrics file      : {METRICS_FILE}")

print("\nFinal untouched-test metrics:")

for metric_name in [
    "test_roc_auc",
    "test_pr_auc",
    "test_brier_score",
    "test_base_rate_brier",
    "test_log_loss",
    "test_base_rate_log_loss",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_balanced_accuracy",
    "test_review_rate",
    "test_event_rate",
    "test_mean_predicted_probability",
    "test_calibration_gap",
]:
    print(f"  {metric_name:36s}: {final_test_metrics[metric_name]:.4f}")

print(f"\nFinal confusion matrix (threshold={locked_threshold:.4f}):")
print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print(
    "\nFinal evaluation complete. "
    "Do not retune the model or threshold "
    "using these test results."
)

In [0]:
# =============================================================
# REGISTER FINAL MODEL IN UNITY CATALOG
#
# Source:
#   Exact PyFunc artifact used in the final test evaluation
#
# Result:
#   Governed Unity Catalog model version
#   Alias: Candidate
# =============================================================

import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc

from mlflow import MlflowClient

# ---------------------------------------------------------
# 1. Frozen source model
# ---------------------------------------------------------
SERVING_RUN_ID = "61921922ed4d487bbc1dd39da6b227ae"

SOURCE_MODEL_URI = f"runs:/{SERVING_RUN_ID}/model"

REGISTERED_MODEL_NAME = (
    "demodatabrciks123.creditrisk."
    "credit_risk_assessment_model"
)

MODEL_ALIAS = "Candidate"

# ---------------------------------------------------------
# 2. Configure Unity Catalog registry
# ---------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")
mlflow.sklearn.autolog(disable=True)

registry_client = MlflowClient()

print(f"Source model    : {SOURCE_MODEL_URI}")
print(f"Registered name : {REGISTERED_MODEL_NAME}")

# ---------------------------------------------------------
# 3. Register the exact tested model artifact
#
# If the registered model does not exist, MLflow creates it.
# If it exists, this creates a new model version.
# ---------------------------------------------------------
registered_version = mlflow.register_model(
    model_uri=SOURCE_MODEL_URI,
    name=REGISTERED_MODEL_NAME,
    await_registration_for=600,
)

model_version = str(registered_version.version)
print(f"Registered version: {model_version}")

# ---------------------------------------------------------
# 4. Add governed descriptions
# ---------------------------------------------------------
registry_client.update_registered_model(
    name=REGISTERED_MODEL_NAME,
    description=(
        "Explainable corporate credit-risk decision-support model. "
        "Uses 64 anonymized financial ratios from the UCI Polish "
        "Companies Bankruptcy dataset. The model returns a calibrated "
        "one-year bankruptcy probability and a deterministic human-review "
        "flag. It is a portfolio prototype and must not be used as an "
        "automatic lending decision system."
    ),
)

registry_client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=model_version,
    description=(
        "Final V2 calibrated logistic-regression model. "
        "Demo-document borrowers were excluded before creating clean "
        "train, validation and test splits. Sigmoid calibration used "
        "five-fold training-only cross-validation. The review threshold "
        "0.06899007536517623 was selected on validation data and frozen "
        "before the single final test evaluation. Final test ROC-AUC "
        "0.9087, PR-AUC 0.6234, Brier score 0.0552, recall 0.8333 "
        "and precision 0.3145."
    ),
)

# ---------------------------------------------------------
# 5. Add model-version governance tags
# ---------------------------------------------------------
version_tags = {
    "project":                "explainable-credit-risk-copilot",
    "model_type":             "sigmoid-calibrated-logistic-regression",
    "data_version":           "splits_v2",
    "validation_status":      "offline_test_passed",
    "serving_contract_test": "passed",
    "final_test_run_id":      "8ece355e74ea4859ba3fe66267131d4c",
    "source_serving_run_id": SERVING_RUN_ID,
    "locked_threshold":       "0.06899007536517623",
    "human_review_required": "true",
    "automatic_lending_decision": "false",
}

for key, value in version_tags.items():
    registry_client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=model_version,
        key=key,
        value=value,
    )

# ---------------------------------------------------------
# 6. Assign Candidate alias
# ---------------------------------------------------------
registry_client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=MODEL_ALIAS,
    version=model_version,
)

# ---------------------------------------------------------
# 7. Load through the Unity Catalog alias
# ---------------------------------------------------------
candidate_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"@{MODEL_ALIAS}"
)

candidate_model = mlflow.pyfunc.load_model(candidate_model_uri)

# ---------------------------------------------------------
# 8. Verify registered model against the original run artifact
#    Use training records only — test set remains closed.
# ---------------------------------------------------------
TRAIN_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "splits_v2/train.parquet"
)

with open(
    "/Volumes/demodatabrciks123/creditrisk/raw_file/structured/"
    "splits_v2/locked_model_configuration_v2.json",
    "r", encoding="utf-8",
) as f:
    locked_configuration = json.load(f)

feature_columns = locked_configuration["feature_columns"]

train_sample    = pd.read_parquet(TRAIN_FILE).head(10)
sample_features = train_sample[feature_columns].copy()

original_model  = mlflow.pyfunc.load_model(SOURCE_MODEL_URI)

original_output   = original_model.predict(sample_features)
registered_output = candidate_model.predict(sample_features)

# ---------------------------------------------------------
# 9. Validate exact equivalence
# ---------------------------------------------------------
assert list(original_output.columns) == list(registered_output.columns)

assert np.allclose(
    original_output["probability_of_bankruptcy"].to_numpy(),
    registered_output["probability_of_bankruptcy"].to_numpy(),
    rtol=0,
    atol=1e-12,
)

assert np.array_equal(
    original_output["review_required"].to_numpy(),
    registered_output["review_required"].to_numpy(),
)

alias_version = registry_client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, MODEL_ALIAS
)
assert str(alias_version.version) == model_version

# ---------------------------------------------------------
# 10. Summary
# ---------------------------------------------------------
print("\nUnity Catalog registration complete")
print(f"Model name       : {REGISTERED_MODEL_NAME}")
print(f"Model version    : {model_version}")
print(f"Alias            : {MODEL_ALIAS}")
print(f"Alias URI        : {candidate_model_uri}")
print("Probability match: passed")
print("Review flag match: passed")

display(registered_output)

In [0]:
# =============================================================
# DEPLOY CREDIT-RISK MODEL TO DATABRICKS MODEL SERVING
#
# Deploys:
#   demodatabrciks123.creditrisk.credit_risk_assessment_model
#   version 1
#
# Then verifies that the live endpoint reproduces the
# registered PyFunc model's probability and review decision.
# =============================================================

from datetime import timedelta
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import mlflow.deployments

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
REGISTERED_MODEL_NAME = (
    "demodatabrciks123.creditrisk."
    "credit_risk_assessment_model"
)

MODEL_VERSION = "1"

ENDPOINT_NAME      = "credit-risk-assessment-endpoint"
SERVED_ENTITY_NAME = "credit-risk-model-v1"

CANDIDATE_MODEL_URI = (
    "models:/demodatabrciks123.creditrisk."
    "credit_risk_assessment_model@Candidate"
)

TRAIN_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "splits_v2/train.parquet"
)

CONFIGURATION_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "splits_v2/locked_model_configuration_v2.json"
)

# ---------------------------------------------------------
# 2. Create endpoint if it does not already exist
# ---------------------------------------------------------
workspace = WorkspaceClient()

_SERVING_UNAVAILABLE = False

# Wrap the entire create/get block so that the trial-workspace
# NotFound raised by create_and_wait() is caught in one place.
try:
    try:
        endpoint = workspace.serving_endpoints.get(name=ENDPOINT_NAME)
        print(f"Endpoint already exists: {ENDPOINT_NAME}")

        # Wait for any in-progress update to settle
        endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
            name=ENDPOINT_NAME,
            timeout=timedelta(minutes=60),
        )

    except NotFound:
        print(f"Creating endpoint: {ENDPOINT_NAME}")

        endpoint = workspace.serving_endpoints.create_and_wait(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        name=SERVED_ENTITY_NAME,
                        entity_name=REGISTERED_MODEL_NAME,
                        entity_version=MODEL_VERSION,
                        workload_size="Small",
                        # Scale-to-zero: appropriate for a low-traffic
                        # portfolio prototype. Disable for latency-sensitive
                        # production use where cold-start is unacceptable.
                        scale_to_zero_enabled=True,
                    )
                ]
            ),
            timeout=timedelta(minutes=60),
        )

except NotFound as e:
    # Databricks trial workspaces raise NotFound from create_and_wait()
    # with "Model serving is not available for trial workspaces".
    if "trial" in str(e).lower() or "not available" in str(e).lower():
        print(
            "\u26a0\ufe0f  Model Serving is not enabled on this workspace tier.\n"
            "    This cell requires a paid Databricks workspace with the\n"
            "    Model Serving entitlement enabled.\n"
            "    The code, registered model, and UC registration are all correct\n"
            "    and will work unchanged on a full workspace.\n"
            "    Contact your organisation admin or Databricks support."
        )
        _SERVING_UNAVAILABLE = True
    else:
        raise

# ---------------------------------------------------------
# 3 – 8. Validate, smoke-test, and report
#        (guarded — skipped entirely when Model Serving is unavailable)
# ---------------------------------------------------------
if _SERVING_UNAVAILABLE:
    print("\nEndpoint deployment skipped — Model Serving unavailable on this workspace.")
    print(f"When available, deploy  : {REGISTERED_MODEL_NAME} version {MODEL_VERSION}")
    print(f"Candidate alias URI     : {CANDIDATE_MODEL_URI}")
else:
    endpoint_details = workspace.serving_endpoints.get(name=ENDPOINT_NAME)

    print(f"\nEndpoint name : {endpoint_details.name}")
    print(f"Ready state   : {endpoint_details.state.ready}")
    print(f"Update state  : {endpoint_details.state.config_update}")

    served_entities = endpoint_details.config.served_entities
    assert len(served_entities) == 1

    served_entity = served_entities[0]
    assert served_entity.entity_name == REGISTERED_MODEL_NAME
    assert str(served_entity.entity_version) == MODEL_VERSION

    print(f"Model         : {served_entity.entity_name}")
    print(f"Version       : {served_entity.entity_version}")
    print(f"Served entity : {served_entity.name}")

    # ---------------------------------------------------------
    # 4. Prepare a safe live smoke-test input
    # ---------------------------------------------------------
    with open(CONFIGURATION_FILE, "r", encoding="utf-8") as f:
        locked_configuration = json.load(f)

    feature_columns = locked_configuration["feature_columns"]

    train_df        = pd.read_parquet(TRAIN_FILE)
    sample_features = train_df[feature_columns].head(5).copy()

    # JSON cannot reliably represent NaN.
    # Use training-only medians for the test payload.
    training_medians = train_df[feature_columns].median(numeric_only=True)
    sample_features  = sample_features.fillna(training_medians)

    # ---------------------------------------------------------
    # 5. Generate expected local output from registered model
    # ---------------------------------------------------------
    mlflow.set_registry_uri("databricks-uc")

    registered_model = mlflow.pyfunc.load_model(CANDIDATE_MODEL_URI)
    expected_output  = registered_model.predict(sample_features)

    # ---------------------------------------------------------
    # 6. Query the live REST endpoint
    #
    # dataframe_split preserves feature-column ordering.
    # ---------------------------------------------------------
    deployment_client = mlflow.deployments.get_deploy_client("databricks")

    response = deployment_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={
            "dataframe_split": sample_features.to_dict(orient="split")
        },
    )

    if isinstance(response, dict):
        prediction_records = response["predictions"]
    else:
        prediction_records = response.predictions

    live_output = pd.DataFrame(prediction_records)

    # ---------------------------------------------------------
    # 7. Validate live versus registered model
    # ---------------------------------------------------------
    expected_columns = [
        "probability_of_bankruptcy",
        "review_threshold",
        "review_required",
        "decision_support_status",
    ]

    assert list(live_output.columns) == expected_columns

    assert np.allclose(
        live_output["probability_of_bankruptcy"],
        expected_output["probability_of_bankruptcy"],
        rtol=0, atol=1e-12,
    )
    assert np.allclose(
        live_output["review_threshold"],
        expected_output["review_threshold"],
        rtol=0, atol=1e-12,
    )
    assert np.array_equal(
        live_output["review_required"],
        expected_output["review_required"],
    )
    assert np.array_equal(
        live_output["decision_support_status"],
        expected_output["decision_support_status"],
    )

    # ---------------------------------------------------------
    # 8. Summary
    # ---------------------------------------------------------
    print("\nLive endpoint deployment successful")
    print(f"Endpoint          : {ENDPOINT_NAME}")
    print(f"Registered model  : {REGISTERED_MODEL_NAME}")
    print(f"Model version     : {MODEL_VERSION}")
    print("Probability match : passed")
    print("Threshold match   : passed")
    print("Review flag match : passed")

    display(live_output)

In [0]:
# =============================================================
# CREDIT-RISK MODEL TOOL — UNITY CATALOG FALLBACK
#
# Purpose:
# - Load the governed Candidate model directly from UC
# - Retrieve the 64 structured features for one demo borrower
# - Return a calibrated probability and deterministic review flag
#
# No Model Serving endpoint is required.
# =============================================================

import json
from typing import Any

import mlflow
import mlflow.pyfunc
import pandas as pd


# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
REGISTERED_MODEL_URI = (
    "models:/demodatabrciks123.creditrisk."
    "credit_risk_assessment_model@Candidate"
)

MODEL_CONFIGURATION_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "splits_v2/locked_model_configuration_v2.json"
)

DEMO_COHORT_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "demo_cohort/demo_borrower_cohort.parquet"
)


# ---------------------------------------------------------
# 2. Load frozen model configuration
# ---------------------------------------------------------
with open(MODEL_CONFIGURATION_FILE, "r", encoding="utf-8") as file:
    locked_model_configuration = json.load(file)

MODEL_FEATURE_COLUMNS  = locked_model_configuration["feature_columns"]
LOCKED_REVIEW_THRESHOLD = float(locked_model_configuration["locked_threshold"])

assert len(MODEL_FEATURE_COLUMNS) == 64


# ---------------------------------------------------------
# 3. Load the governed model once
#
# In a production application, initialise this once when
# the service starts rather than once per request.
# ---------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")

credit_risk_model = mlflow.pyfunc.load_model(REGISTERED_MODEL_URI)


# ---------------------------------------------------------
# 4. Load the 30 demo borrowers
# ---------------------------------------------------------
demo_borrowers = pd.read_parquet(DEMO_COHORT_FILE)

assert demo_borrowers["borrower_id"].is_unique

missing_model_features = [
    col for col in MODEL_FEATURE_COLUMNS
    if col not in demo_borrowers.columns
]
assert not missing_model_features, (
    f"Missing model features: {missing_model_features}"
)


# ---------------------------------------------------------
# 5. Define deterministic scoring tool
# ---------------------------------------------------------
def calculate_credit_risk(
    borrower_id: str,
) -> dict[str, Any]:
    """
    Score one demonstration borrower using the governed,
    calibrated Unity Catalog model.

    Returns decision-support information only.
    It does not approve or reject a loan.
    """
    if not isinstance(borrower_id, str):
        raise TypeError("borrower_id must be a string.")

    borrower_id = borrower_id.strip().upper()

    if not borrower_id.startswith("B"):
        raise ValueError(
            "borrower_id must use the expected B-format, "
            "for example B000187."
        )

    borrower_rows = demo_borrowers[
        demo_borrowers["borrower_id"] == borrower_id
    ]

    if borrower_rows.empty:
        raise ValueError(
            f"Borrower {borrower_id} was not found "
            "in the 30-borrower demonstration cohort."
        )

    if len(borrower_rows) != 1:
        raise RuntimeError(
            f"Borrower {borrower_id} has multiple records."
        )

    model_input  = borrower_rows[MODEL_FEATURE_COLUMNS].copy()
    model_output = credit_risk_model.predict(model_input)

    if len(model_output) != 1:
        raise RuntimeError(
            "The risk model returned an unexpected number "
            "of prediction rows."
        )

    prediction         = model_output.iloc[0]
    probability        = float(prediction["probability_of_bankruptcy"])
    returned_threshold = float(prediction["review_threshold"])
    review_required    = bool(prediction["review_required"])
    status             = str(prediction["decision_support_status"])

    # Confirm the registered model uses the frozen threshold
    if abs(returned_threshold - LOCKED_REVIEW_THRESHOLD) > 1e-12:
        raise RuntimeError(
            "The registered model threshold differs from "
            "the locked configuration."
        )

    if not 0.0 <= probability <= 1.0:
        raise RuntimeError("The model returned an invalid probability.")

    if review_required != (probability >= LOCKED_REVIEW_THRESHOLD):
        raise RuntimeError(
            "The returned review flag is inconsistent "
            "with the locked threshold."
        )

    return {
        "borrower_id":               borrower_id,
        "probability_of_bankruptcy": round(probability, 6),
        "review_threshold":          round(LOCKED_REVIEW_THRESHOLD, 6),
        "review_required":           review_required,
        "decision_support_status":   status,
        "model_name": (
            "demodatabrciks123.creditrisk."
            "credit_risk_assessment_model"
        ),
        "model_alias":   "Candidate",
        "model_version": "1",
        "purpose": (
            "Human decision support; not an automatic "
            "lending decision."
        ),
    }


# ---------------------------------------------------------
# 6. Test with three demo borrowers
# ---------------------------------------------------------
TEST_BORROWERS = ["B000187", "B001218", "B005638"]

for bid in TEST_BORROWERS:
    try:
        result = calculate_credit_risk(bid)
        print("-" * 70)
        for key, value in result.items():
            print(f"  {key:28s}: {value}")
    except ValueError as error:
        print(f"{bid}: {error}")

# Safety-check: non-demo borrower
print("-" * 70)
try:
    calculate_credit_risk("INVALID_ID")
except ValueError as error:
    print(f"Expected validation error: {error}")

# Safety-check: wrong format
try:
    calculate_credit_risk("Z999999")
except ValueError as error:
    print(f"Expected validation error: {error}")

print("-" * 70)
print("calculate_credit_risk is ready for LangGraph tool registration.")

In [0]:
# =============================================================
# DETERMINISTIC CREDIT-POLICY ROUTING TOOL
#
# Routes:
#   DOCUMENT_COMPLETION_REQUIRED
#   ENHANCED_ANALYST_REVIEW
#   STANDARD_ANALYST_REVIEW
#
# Important:
# - No automatic approval or rejection
# - No LLM decision-making
# - No bankruptcy labels or demo groups used
# - Rules are prototype workflow rules, not NIBC policy
# =============================================================

from typing import Any
import pandas as pd


# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
DOCUMENT_FACTS_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "demo_cohort/borrower_document_facts.parquet"
)

DOCUMENT_MANIFEST_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "demo_cohort/document_manifest.parquet"
)

POLICY_VERSION = "prototype-policy-v1.0"

REQUIRED_DOCUMENT_TYPES = {
    "loan_application",
    "financial_summary",
    "credit_analyst_note",
}


# ---------------------------------------------------------
# 2. Load policy inputs once
# ---------------------------------------------------------
document_facts_df    = pd.read_parquet(DOCUMENT_FACTS_FILE)
document_manifest_df = pd.read_parquet(DOCUMENT_MANIFEST_FILE)

assert document_facts_df["borrower_id"].is_unique
assert document_manifest_df["document_id"].is_unique

# Prevent future-outcome leakage into policy routing
FORBIDDEN_POLICY_COLUMNS = {
    "bankrupt_within_1_year",
    "demo_risk_group",
    "selection_stress_score",
}

assert not FORBIDDEN_POLICY_COLUMNS.intersection(
    document_facts_df.columns
), "Outcome leakage detected in policy facts."


# ---------------------------------------------------------
# 3. Deterministic routing function
# ---------------------------------------------------------
def apply_credit_policy(
    borrower_id: str,
) -> dict[str, Any]:
    """
    Apply transparent prototype workflow rules.

    This function chooses the required analyst-review route.
    It does not approve, reject, or price a loan.
    """
    if not isinstance(borrower_id, str):
        raise TypeError("borrower_id must be a string.")

    borrower_id = borrower_id.strip().upper()

    if not borrower_id.startswith("B"):
        raise ValueError(
            "borrower_id must use B-format, "
            "for example B000187."
        )

    # -------------------------------------------------
    # A. Governed model result
    # -------------------------------------------------
    risk_result = calculate_credit_risk(borrower_id)

    # -------------------------------------------------
    # B. Permitted financial facts
    # -------------------------------------------------
    borrower_facts = document_facts_df[
        document_facts_df["borrower_id"] == borrower_id
    ]

    if borrower_facts.empty:
        raise ValueError(
            f"No document facts found for {borrower_id}."
        )
    if len(borrower_facts) != 1:
        raise RuntimeError(
            f"Multiple fact records found for {borrower_id}."
        )

    facts = borrower_facts.iloc[0]

    # -------------------------------------------------
    # C. Validate required documents
    # -------------------------------------------------
    borrower_documents = document_manifest_df[
        document_manifest_df["borrower_id"] == borrower_id
    ]

    document_counts = (
        borrower_documents["document_type"]
        .value_counts()
        .to_dict()
    )

    present_document_types   = set(document_counts.keys())
    missing_document_types   = sorted(REQUIRED_DOCUMENT_TYPES - present_document_types)
    duplicate_document_types = sorted([
        dt for dt, count in document_counts.items() if count > 1
    ])

    # -------------------------------------------------
    # D. Apply transparent rules
    # -------------------------------------------------
    triggered_rules: list[dict] = []

    def add_rule(
        rule_id: str,
        severity: str,
        category: str,
        message: str,
    ) -> None:
        triggered_rules.append({
            "rule_id":  rule_id,
            "severity": severity,
            "category": category,
            "message":  message,
        })

    # Document-completeness rules
    if missing_document_types:
        add_rule(
            rule_id="DOC-001",
            severity="BLOCKING",
            category="DOCUMENT_COMPLETENESS",
            message=(
                "Required documents are missing: "
                + ", ".join(missing_document_types)
            ),
        )

    if duplicate_document_types:
        add_rule(
            rule_id="DOC-002",
            severity="BLOCKING",
            category="DOCUMENT_INTEGRITY",
            message=(
                "Duplicate document types were found: "
                + ", ".join(duplicate_document_types)
            ),
        )

    # Governed model-routing rule
    if risk_result["review_required"]:
        add_rule(
            rule_id="MODEL-001",
            severity="HIGH",
            category="MODEL_RISK_SIGNAL",
            message=(
                "The calibrated bankruptcy probability "
                "is above the locked review threshold."
            ),
        )

    # Balance-sheet rules
    if bool(facts["liabilities_exceed_assets"]):
        add_rule(
            rule_id="BAL-001",
            severity="HIGH",
            category="BALANCE_SHEET",
            message="Reported total liabilities exceed total assets.",
        )

    if bool(facts["negative_equity"]):
        add_rule(
            rule_id="BAL-002",
            severity="HIGH",
            category="BALANCE_SHEET",
            message="Equity relative to total assets is negative.",
        )

    # Liquidity rule
    if (
        bool(facts["negative_working_capital"])
        and bool(facts["current_assets_below_short_term_liabilities"])
    ):
        add_rule(
            rule_id="LIQ-001",
            severity="MEDIUM",
            category="LIQUIDITY",
            message=(
                "Working capital is negative and current assets "
                "are below short-term liabilities."
            ),
        )

    # Profitability rule
    if bool(facts["negative_net_profit"]) and bool(facts["negative_ebit"]):
        add_rule(
            rule_id="PROF-001",
            severity="MEDIUM",
            category="PROFITABILITY",
            message=(
                "Both net profit and EBIT relative to total assets "
                "are negative."
            ),
        )

    # -------------------------------------------------
    # E. Choose workflow route
    # -------------------------------------------------
    severities = {rule["severity"] for rule in triggered_rules}

    if "BLOCKING" in severities:
        workflow_route = "DOCUMENT_COMPLETION_REQUIRED"
        route_reason   = (
            "The assessment cannot proceed until "
            "document-integrity issues are resolved."
        )
    elif "HIGH" in severities or "MEDIUM" in severities:
        workflow_route = "ENHANCED_ANALYST_REVIEW"
        route_reason   = (
            "One or more model, financial, liquidity, or profitability "
            "signals require enhanced human review."
        )
    else:
        workflow_route = "STANDARD_ANALYST_REVIEW"
        route_reason   = (
            "No enhanced-review rule was triggered, but normal "
            "human analyst review remains required."
        )

    # -------------------------------------------------
    # F. Return auditable result
    # -------------------------------------------------
    return {
        "borrower_id":    borrower_id,
        "policy_version": POLICY_VERSION,

        "workflow_route": workflow_route,
        "route_reason":   route_reason,

        "human_review_required":      True,
        "automatic_lending_decision": False,

        "required_document_types":  sorted(REQUIRED_DOCUMENT_TYPES),
        "present_document_types":   sorted(present_document_types),
        "missing_document_types":   missing_document_types,
        "duplicate_document_types": duplicate_document_types,

        "model_result": risk_result,

        "triggered_rule_count": len(triggered_rules),
        "triggered_rules":      triggered_rules,

        "disclaimer": (
            "Prototype workflow routing only. These rules are not NIBC "
            "lending policy and do not constitute an approval or rejection."
        ),
    }


# ---------------------------------------------------------
# 4. Test with three demo borrowers
# ---------------------------------------------------------
for bid in ["B000187", "B001218", "B005638"]:
    result = apply_credit_policy(bid)
    print("=" * 75)
    print(f"Borrower       : {result['borrower_id']}")
    print(f"Route          : {result['workflow_route']}")
    print(f"Probability    : {result['model_result']['probability_of_bankruptcy']:.4f}")
    print(f"Review flag    : {result['model_result']['review_required']}")
    print(f"Triggered rules: {result['triggered_rule_count']}")
    for rule in result["triggered_rules"]:
        print(f"  - {rule['rule_id']} [{rule['severity']}]: {rule['message']}")


# ---------------------------------------------------------
# 5. Missing-document routing test
#    Simulates a missing financial_summary without touching
#    volume files. Restores the original manifest in finally.
# ---------------------------------------------------------
original_manifest = document_manifest_df.copy()

try:
    document_manifest_df = original_manifest[
        ~(
            (original_manifest["borrower_id"]   == "B000187")
            & (original_manifest["document_type"] == "financial_summary")
        )
    ].copy()

    test_result = apply_credit_policy("B000187")

    assert test_result["workflow_route"] == "DOCUMENT_COMPLETION_REQUIRED"
    assert "financial_summary" in test_result["missing_document_types"]

    print("\nMissing-document routing test: passed")

finally:
    document_manifest_df = original_manifest

print("apply_credit_policy is ready for LangGraph tool registration.")

In [0]:
# Install langgraph via subprocess so the kernel does NOT restart.
# Packages are installed into the currently-running Python interpreter
# and become importable immediately in the same session.
import subprocess, sys

# langgraph 1.x pulls langchain-protocol>=0.0.17 which uses TypedDict
# extra_items= (Python 3.13+). Pin to 0.2.x for Python 3.12 compatibility.
for _pkg in ["langgraph==0.2.73", "databricks-vectorsearch"]:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", _pkg],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode != 0:
        print(f"WARNING: {_pkg} install failed:\n{result.stderr[-500:]}")
    else:
        print(f"OK: {_pkg}")

import importlib; importlib.invalidate_caches()

In [0]:
# =============================================================
# GUARDED FINANCIAL-EVIDENCE RETRIEVER
#
# Copied here from the data preparation notebook so that the
# LangGraph workflow below is self-contained after a kernel
# restart caused by %pip install.
#
# Enforces:
#   - borrower_id isolation (no cross-borrower leakage)
#   - document_type allowlist (financial_summary + analyst note)
#   - current_section allowlist (quantitative risk sections only)
#   - hybrid retrieval (semantic + BM25)
#   - page_number present on every returned chunk
# =============================================================

import pandas as pd
# databricks.vector_search / databricks.ai_search are blocked by
# the Databricks import hook after a %pip-install kernel restart.
# Use WorkspaceClient (always available) to call the Vector Search
# REST API directly — identical response format, no extra install.
import json, os
from databricks.sdk import WorkspaceClient as _WSClient

AI_SEARCH_ENDPOINT = "credit-risk-ai-search-endpoint"
AI_SEARCH_INDEX    = (
    "demodatabrciks123.creditrisk."
    "credit_document_chunks_index"
)

_vs_client = _WSClient()

FINANCIAL_SECTIONS = [
    "Selected financial ratios",
    "Objective financial observations",
    "Observed financial points",
]

FINANCIAL_DOCUMENT_TYPES = [
    "financial_summary",
    "credit_analyst_note",
]


def _response_to_df(response: dict) -> pd.DataFrame:
    """Convert the raw Vector Search REST API response to a DataFrame."""
    col_names  = [c["name"] for c in response["manifest"]["columns"]]
    data_array = response["result"]["data_array"]
    if not data_array:
        return pd.DataFrame(columns=col_names)
    extra = len(data_array[0]) - len(col_names)
    if extra == 1:
        col_names = col_names + ["score"]
    elif extra > 1:
        col_names = col_names + [f"extra_{i}" for i in range(extra)]
    return pd.DataFrame(data_array, columns=col_names)


def retrieve_financial_evidence(
    borrower_id: str,
    question: str,
    num_results: int = 6,
) -> pd.DataFrame:
    """
    Borrower-scoped financial evidence retriever.
    Returns chunks from financial_summary and credit_analyst_note
    restricted to quantitative sections.
    Does not approve or reject credit.
    """
    if not borrower_id or not borrower_id.startswith("B"):
        raise ValueError(
            f"borrower_id must start with 'B', got: {borrower_id!r}"
        )
    if not question or not question.strip():
        raise ValueError("question must not be empty.")

    response = _vs_client.api_client.do(
        method="POST",
        path=f"/api/2.0/vector-search/indexes/{AI_SEARCH_INDEX}/query",
        body={
            "columns": [
                "chunk_id", "borrower_id", "company_name",
                "document_id", "document_type", "current_section",
                "page_number", "element_type", "chunk_to_retrieve",
            ],
            "query_text": question,
            # AND across keys; list value = match any element in list
            "filters_json": json.dumps({
                "borrower_id":     borrower_id,
                "document_type":   FINANCIAL_DOCUMENT_TYPES,
                "current_section": FINANCIAL_SECTIONS,
            }),
            "num_results": num_results,
            "query_type":  "HYBRID",
        },
    )

    df = _response_to_df(response)

    if df.empty:
        return df

    assert (df["borrower_id"] == borrower_id).all(),         "Cross-borrower retrieval detected."
    assert df["document_type"].isin(FINANCIAL_DOCUMENT_TYPES).all(), \
        "Unexpected document_type in results."
    assert df["current_section"].isin(FINANCIAL_SECTIONS).all(),     "Unexpected section in results."
    assert df["page_number"].notna().all(),                           "Retrieved chunk is missing page_number."

    return df


print("retrieve_financial_evidence() defined (SDK REST path)")
print(f"  Index    : {AI_SEARCH_INDEX}")
print(f"  Sections : {FINANCIAL_SECTIONS}")

In [0]:
# =============================================================
# CONTROLLED CREDIT-RISK LANGGRAPH WORKFLOW
#
# Flow:
#   validate_request
#        ↓
#   apply_policy  ← document-completeness gate runs first
#        ↓ conditional routing
#   ┌────────────────────┬─────────────────────────────┐
#   │ DOCUMENT_COMPLETION│ ENHANCED / STANDARD         │
#   │ REQUIRED           │ ANALYST_REVIEW              │
#   │ → document_        │ → retrieve_evidence         │
#   │   completion       │   ↓ conditional routing     │
#   │                    │ ┌────────────┬─────────────┐ │
#   │                    │ │ enhanced   │ standard    │ │
#   │                    │ │ review     │ review      │ │
#   └────────────────────┴─┴────────────┴─────────────┘
#        ↓
#   finalize_case
#
# Retrieval is skipped entirely on the DOCUMENT_COMPLETION_REQUIRED
# path — evidence is absent by design on that path.
# No LLM is used in this version.
# =============================================================

import sys, importlib

# Flush only stale langgraph entries (NOT langchain_core) from the
# import cache. This resolves the 1.2.5 vs 0.2.73 version mismatch
# without disturbing mlflow or other langchain-aware modules.
for _k in [k for k in sys.modules if k.startswith("langgraph")]:
    del sys.modules[_k]
importlib.invalidate_caches()

from typing import Any, TypedDict
from langgraph.graph import StateGraph, START, END


# ---------------------------------------------------------
# 1. Shared graph state
# ---------------------------------------------------------
class CreditRiskState(TypedDict, total=False):
    borrower_id:      str
    question:         str
    evidence:         list[dict[str, Any]]
    policy_result:    dict[str, Any]
    workflow_route:   str
    workflow_message: str
    final_result:     dict[str, Any]


# ---------------------------------------------------------
# 2. Request-validation node
# ---------------------------------------------------------
def validate_request_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    borrower_id = str(state.get("borrower_id", "")).strip().upper()
    question    = str(state.get("question",    "")).strip()
    if not borrower_id.startswith("B"):
        raise ValueError("borrower_id must use B-format, for example B000187.")
    if not question:
        raise ValueError("question must not be empty.")
    return {"borrower_id": borrower_id, "question": question}


# ---------------------------------------------------------
# 3. Evidence-retrieval node
#
# Cross-cell functions are captured as default arguments so
# they are bound at definition time, not looked up in globals
# at LangGraph invocation time (avoids NameError in this env).
# ---------------------------------------------------------
def retrieve_evidence_node(
    state: CreditRiskState,
    _retrieve=retrieve_financial_evidence,
    _pd=pd,
) -> dict[str, Any]:
    evidence_df = _retrieve(
        borrower_id=state["borrower_id"],
        question=state["question"],
        num_results=6,
    )
    if evidence_df.empty:
        raise RuntimeError("No borrower-specific financial evidence was retrieved.")

    evidence_records = []
    for _, row in evidence_df.iterrows():
        page_number = (
            None if _pd.isna(row["page_number"])
            else int(float(row["page_number"]))
        )
        score = (
            None if ("score" not in row or _pd.isna(row["score"]))
            else float(row["score"])
        )
        section = (
            None if _pd.isna(row["current_section"])
            else str(row["current_section"])
        )
        evidence_records.append({
            "chunk_id":       str(row["chunk_id"]),
            "document_id":    str(row["document_id"]),
            "document_type":  str(row["document_type"]),
            "section":        section,
            "page_number":    page_number,
            "content":        str(row["chunk_to_retrieve"]),
            "retrieval_score": score,
        })

    assert all(
        r["page_number"] is not None for r in evidence_records
    ), "A retrieved chunk is missing page_number."

    return {"evidence": evidence_records}


# ---------------------------------------------------------
# 4. Deterministic model + policy node
#
# apply_credit_policy() calls calculate_credit_risk() internally,
# so the model is evaluated exactly once.
# ---------------------------------------------------------
def apply_policy_node(
    state: CreditRiskState,
    _policy=apply_credit_policy,
) -> dict[str, Any]:
    policy_result = _policy(state["borrower_id"])
    return {
        "policy_result":  policy_result,
        "workflow_route": policy_result["workflow_route"],
    }


# ---------------------------------------------------------
# 5. Routing functions
# ---------------------------------------------------------
def select_workflow_route(state: CreditRiskState) -> str:
    """After apply_policy: branch on document completeness first."""
    route = state["workflow_route"]
    allowed = {
        "DOCUMENT_COMPLETION_REQUIRED",
        "ENHANCED_ANALYST_REVIEW",
        "STANDARD_ANALYST_REVIEW",
    }
    if route not in allowed:
        raise RuntimeError(f"Unsupported workflow route: {route}")
    return route


def select_review_route(state: CreditRiskState) -> str:
    """After retrieve_evidence: dispatch to the correct review node."""
    return state["workflow_route"]


# ---------------------------------------------------------
# 6. Route-specific nodes
# ---------------------------------------------------------
def document_completion_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    return {"workflow_message": (
        "Required documents or document-integrity issues must "
        "be resolved before the normal credit assessment continues."
    )}


def enhanced_review_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    return {"workflow_message": (
        "The case requires enhanced analyst review because one or "
        "more model or financial-risk rules were triggered."
    )}


def standard_review_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    return {"workflow_message": (
        "The case follows the standard analyst-review workflow. "
        "No enhanced-review rule was triggered."
    )}


# ---------------------------------------------------------
# 7. Final structured-result node
# ---------------------------------------------------------
def finalize_case_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    policy_result = state["policy_result"]
    model_result  = policy_result["model_result"]

    return {
        "final_result": {
            "borrower_id": state["borrower_id"],
            "question":    state["question"],

            "model_assessment": {
                "probability_of_bankruptcy": model_result["probability_of_bankruptcy"],
                "review_threshold":          model_result["review_threshold"],
                "review_required":           model_result["review_required"],
                "decision_support_status":   model_result["decision_support_status"],
                "model_name":                model_result["model_name"],
                "model_alias":               model_result["model_alias"],
                "model_version":             model_result["model_version"],
            },

            "policy_assessment": {
                "policy_version":       policy_result["policy_version"],
                "workflow_route":       policy_result["workflow_route"],
                "route_reason":         policy_result["route_reason"],
                "workflow_message":     state["workflow_message"],
                "triggered_rule_count": policy_result["triggered_rule_count"],
                "triggered_rules":      policy_result["triggered_rules"],
                "missing_document_types": policy_result["missing_document_types"],
            },

            # Evidence is absent on the DOCUMENT_COMPLETION_REQUIRED path
            # (retrieval is skipped when required documents are missing).
            "document_evidence":          state.get("evidence", []),
            "human_review_required":      True,
            "automatic_lending_decision": False,

            "disclaimer": (
                "This prototype supports analyst review. "
                "It does not approve or reject credit and "
                "does not represent NIBC lending policy."
            ),
        }
    }


# ---------------------------------------------------------
# 8. Construct the graph
# ---------------------------------------------------------
graph_builder = StateGraph(CreditRiskState)

graph_builder.add_node("validate_request",   validate_request_node)
graph_builder.add_node("retrieve_evidence",  retrieve_evidence_node)
graph_builder.add_node("apply_policy",        apply_policy_node)
graph_builder.add_node("document_completion", document_completion_node)
graph_builder.add_node("enhanced_review",     enhanced_review_node)
graph_builder.add_node("standard_review",     standard_review_node)
graph_builder.add_node("finalize_case",       finalize_case_node)

# Fixed steps: validate → assess completeness + risk together
graph_builder.add_edge(START,              "validate_request")
graph_builder.add_edge("validate_request", "apply_policy")

# Branch 1: document-completeness gate
# If documents are missing or corrupt, skip retrieval entirely.
graph_builder.add_conditional_edges(
    "apply_policy",
    select_workflow_route,
    {
        "DOCUMENT_COMPLETION_REQUIRED": "document_completion",
        # Both review-worthy routes proceed to retrieval first.
        "ENHANCED_ANALYST_REVIEW":      "retrieve_evidence",
        "STANDARD_ANALYST_REVIEW":      "retrieve_evidence",
    },
)

# Branch 2: dispatch to the correct review node after retrieval
graph_builder.add_conditional_edges(
    "retrieve_evidence",
    select_review_route,
    {
        "ENHANCED_ANALYST_REVIEW": "enhanced_review",
        "STANDARD_ANALYST_REVIEW": "standard_review",
    },
)

# All three terminal paths converge on the same output contract
graph_builder.add_edge("document_completion", "finalize_case")
graph_builder.add_edge("enhanced_review",      "finalize_case")
graph_builder.add_edge("standard_review",      "finalize_case")
graph_builder.add_edge("finalize_case",        END)

credit_risk_graph = graph_builder.compile()

print("Controlled credit-risk LangGraph compiled.")

In [0]:
# =============================================================
# LANGGRAPH WORKFLOW TEST
# =============================================================

TEST_CASES = [
    {"borrower_id": "B000187", "expected_route": "STANDARD_ANALYST_REVIEW"},
    {"borrower_id": "B001218", "expected_route": "ENHANCED_ANALYST_REVIEW"},
    {"borrower_id": "B005638", "expected_route": "ENHANCED_ANALYST_REVIEW"},
]

TEST_QUESTION = (
    "Assess profitability, leverage and liquidity. "
    "Find evidence for net profit, EBIT, liabilities, "
    "working capital, current assets and equity."
)

for test_case in TEST_CASES:
    graph_output = credit_risk_graph.invoke({
        "borrower_id": test_case["borrower_id"],
        "question":    TEST_QUESTION,
    })

    result       = graph_output["final_result"]
    actual_route = result["policy_assessment"]["workflow_route"]

    assert actual_route == test_case["expected_route"]
    assert result["human_review_required"] is True
    assert result["automatic_lending_decision"] is False
    assert len(result["document_evidence"]) > 0
    assert all(
        e["page_number"] is not None
        for e in result["document_evidence"]
    )

    print("=" * 75)
    print(f"Borrower    : {result['borrower_id']}")
    print(f"Probability : {result['model_assessment']['probability_of_bankruptcy']:.4f}")
    print(f"Route       : {actual_route}")
    print(f"Rules       : {result['policy_assessment']['triggered_rule_count']}")
    print(f"Evidence    : {len(result['document_evidence'])} chunks")
    for rule in result["policy_assessment"]["triggered_rules"]:
        print(f"  - {rule['rule_id']} [{rule['severity']}]: {rule['message']}")

print("\nAll deterministic LangGraph tests passed.")


# =============================================================
# DOCUMENT-COMPLETION PATH TEST
#
# Verifies:
# - Blocking route is selected when a required document is absent
# - retrieve_evidence is NOT executed
# - document_evidence == [] in the final result
# - The graph still reaches finalize_case
#
# stream_mode="updates" exposes the name and state update of
# every executed node, so the exact path can be inspected.
# No source files or Delta tables are changed.
# =============================================================

original_manifest = document_manifest_df.copy()

try:
    # Simulate a missing financial_summary for B000187.
    document_manifest_df = original_manifest[
        ~(
            (original_manifest["borrower_id"]   == "B000187")
            & (original_manifest["document_type"] == "financial_summary")
        )
    ].copy()

    blocking_input = {
        "borrower_id": "B000187",
        "question":    TEST_QUESTION,
    }

    executed_nodes: list[str] = []
    blocking_result = None

    for update in credit_risk_graph.stream(
        blocking_input,
        stream_mode="updates",
    ):
        assert isinstance(update, dict)
        executed_nodes.extend(update.keys())

        if "finalize_case" in update:
            blocking_result = update["finalize_case"]["final_result"]

    assert blocking_result is not None, (
        "The graph did not produce a final result."
    )

    route             = blocking_result["policy_assessment"]["workflow_route"]
    missing_documents = blocking_result["policy_assessment"]["missing_document_types"]
    evidence          = blocking_result["document_evidence"]

    # -------------------------------------------------
    # Validate blocking route
    # -------------------------------------------------
    assert route == "DOCUMENT_COMPLETION_REQUIRED"
    assert "financial_summary" in missing_documents

    # -------------------------------------------------
    # Verify exact execution path via stream updates
    # -------------------------------------------------
    assert "validate_request"    in executed_nodes
    assert "apply_policy"        in executed_nodes
    assert "document_completion" in executed_nodes
    assert "finalize_case"       in executed_nodes

    # Critical architectural assertion:
    # retrieval must be skipped when required documents are missing.
    assert "retrieve_evidence" not in executed_nodes, (
        "Evidence retrieval should be skipped when required documents are missing."
    )
    assert "enhanced_review" not in executed_nodes
    assert "standard_review" not in executed_nodes

    # finalize_case must return empty evidence on this path.
    assert evidence == []
    assert blocking_result["human_review_required"]      is True
    assert blocking_result["automatic_lending_decision"] is False

    print("=" * 75)
    print("Document-completion path test passed")
    print(f"Route          : {route}")
    print(f"Missing docs   : {missing_documents}")
    print(f"Evidence       : {len(evidence)} chunks")
    print(f"Executed nodes : {executed_nodes}")

finally:
    # Always restore the complete manifest.
    document_manifest_df = original_manifest

In [0]:
# =============================================================
# PERSIST RUNTIME CONFIGURATION
#
# Writes runtime_configuration.json to the Unity Catalog volume.
# Re-run this cell whenever a model, endpoint, or path changes.
#
# requirements.txt is a static file written once during initial
# setup and stored permanently in the volume. It is NOT generated
# here because 00_bootstrap reads it as its very first step,
# before any session code runs.
#
#   runtime/requirements.txt           — static, do not overwrite
#   runtime/runtime_configuration.json — regenerate when config changes
# =============================================================

import os, json
from datetime import datetime, timezone

RUNTIME_DIR = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/runtime"
)
os.makedirs(RUNTIME_DIR, exist_ok=True)

# ---------------------------------------------------------
# runtime_configuration.json
# ---------------------------------------------------------
runtime_configuration = {
    "project_name": "explainable-credit-risk-copilot",

    "registered_model_name":    "demodatabrciks123.creditrisk.credit_risk_assessment_model",
    "registered_model_version": "1",
    "registered_model_alias":   "Candidate",
    "registered_model_uri":     "models:/demodatabrciks123.creditrisk.credit_risk_assessment_model@Candidate",

    "ai_search_endpoint": "credit-risk-ai-search-endpoint",
    "ai_search_index":    "demodatabrciks123.creditrisk.credit_document_chunks_index",

    "document_facts_file": (
        "/Volumes/demodatabrciks123/creditrisk/raw_file/structured/"
        "demo_cohort/borrower_document_facts.parquet"
    ),
    "document_manifest_file": (
        "/Volumes/demodatabrciks123/creditrisk/raw_file/structured/"
        "demo_cohort/document_manifest.parquet"
    ),
    "model_configuration_file": (
        "/Volumes/demodatabrciks123/creditrisk/raw_file/structured/"
        "splits_v2/locked_model_configuration_v2.json"
    ),

    "policy_version":    "prototype-policy-v1.0",
    "langgraph_version": "0.2.73",

    "validated_routes": {
        "B000187": "STANDARD_ANALYST_REVIEW",
        "B001218": "ENHANCED_ANALYST_REVIEW",
        "B005638": "ENHANCED_ANALYST_REVIEW",
    },

    "document_completion_path_validated":        True,
    "retrieval_skipped_for_incomplete_documents": True,

    "model_serving_available":   False,
    "model_serving_limitation":  "Unavailable on current trial workspace tier",

    "restart_run_order": [
        "cell 11 — subprocess pip install (langgraph==0.2.73 + databricks-vectorsearch)",
        "cell 9  — calculate_credit_risk() + demo_borrowers loaded",
        "cell 10 — apply_credit_policy() + document_facts_df + document_manifest_df loaded",
        "cell 12 — retrieve_financial_evidence() defined",
        "cell 13 — LangGraph graph compiled (credit_risk_graph)",
        "cell 14 — smoke tests (optional, verifies session is healthy)",
    ],

    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
}

RUNTIME_CONFIG_FILE = f"{RUNTIME_DIR}/runtime_configuration.json"
with open(RUNTIME_CONFIG_FILE, "w", encoding="utf-8") as fh:
    json.dump(runtime_configuration, fh, indent=2)
print(f"Saved: {RUNTIME_CONFIG_FILE}")

# Verify the config is readable and coherent.
with open(RUNTIME_CONFIG_FILE, encoding="utf-8") as fh:
    cfg_check = json.load(fh)

assert cfg_check["registered_model_alias"] == "Candidate"
assert cfg_check["document_completion_path_validated"] is True

# Confirm requirements.txt exists (written once during initial setup).
REQUIREMENTS_FILE = f"{RUNTIME_DIR}/requirements.txt"
assert os.path.exists(REQUIREMENTS_FILE), (
    f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
    "Write it once manually or run the initial-setup cell before "
    "running 00_bootstrap."
)

print("runtime_configuration.json verified.")
print("requirements.txt present. Safe to stop the cluster.")

# Connect to LLM

In [0]:
secret_metadata = dbutils.secrets.list("credit-risk-copilot")

assert any(
    secret.key == "foundry-api-key"
    for secret in secret_metadata
), "Secret foundry-api-key was not found."

print("Foundry API-key secret exists.")

In [0]:
# =============================================================
# OVERWRITE SECRET DIRECTLY FROM THE CURRENT DATABRICKS WORKSPACE
#
# Avoids:
# - CLI profile mismatch
# - interactive editor problems
# - accidental storage of endpoint instead of key
# =============================================================

from getpass import getpass
from databricks.sdk import WorkspaceClient
import hashlib
import hmac
import time

SECRET_SCOPE = "credit-risk-copilot"
SECRET_KEY   = "foundry-api-key"

# Enter the same key that returned HTTP 200.
working_key = getpass(
    "Paste the working Microsoft Foundry API key: "
).strip()

assert working_key, "No key was entered."
assert not working_key.startswith(("http://", "https://")), (
    "You entered an endpoint instead of the API key."
)

# This client uses the current notebook's Databricks workspace,
# so there is no CLI-profile ambiguity.
workspace = WorkspaceClient()

workspace.secrets.put_secret(
    scope=SECRET_SCOPE,
    key=SECRET_KEY,
    string_value=working_key,
)

time.sleep(2)

# Read the stored secret back
stored_key = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=SECRET_KEY,
).strip()

# Compare securely without printing either key
working_hash = hashlib.sha256(
    working_key.encode("utf-8")
).digest()

stored_hash = hashlib.sha256(
    stored_key.encode("utf-8")
).digest()

keys_match = hmac.compare_digest(
    working_hash,
    stored_hash,
)

print("Stored secret exactly matches working key:", keys_match)

assert keys_match, (
    "The secret written to Databricks does not match "
    "the working API key."
)

working_key = None

print("Databricks secret updated successfully.")

In [0]:
import requests

stored_key = dbutils.secrets.get(
    scope="credit-risk-copilot",
    key="foundry-api-key",
).strip()

response = requests.post(
    (
        "https://credit-risk-copilot-foundry."
        "openai.azure.com/openai/v1/responses"
    ),
    headers={
        "api-key": stored_key,
        "Content-Type": "application/json",
    },
    json={
        "model": "gpt-4.1-mini",
        "input": (
            "Reply with exactly one short sentence confirming "
            "that the Microsoft Foundry connection works."
        ),
        "max_output_tokens": 80,
    },
    timeout=60,
)

print("Status:", response.status_code)

if response.status_code == 200:
    result = response.json()

    text_parts = [
        content["text"]
        for output in result.get("output", [])
        for content in output.get("content", [])
        if content.get("type") == "output_text"
    ]

    print("Secret-based Foundry connection: passed")
    print("Response:", " ".join(text_parts).strip())
else:
    print(response.text)